# 04 — Interpret `metrics.csv`

This notebook opens the `metrics.csv` produced by `evaluate_coresets.py` and explains every column group: geographic diagnostics, operator fidelity, KRR, and multi-model downstream. It ends with a tiny ranking exercise that mirrors what the championship analysis does.

**Prerequisite:** notebook 02 has been run (produces `metrics.csv`).

**Runtime:** a few seconds.

**Relevant API:**
- [evaluate_coresets script](../docs/api/scripts.md#evaluate_coresets)
- [evaluation API](../docs/api/evaluation.md)

In [ ]:
from pathlib import Path
import pandas as pd

REPO = Path.cwd().resolve()
while REPO != REPO.parent and not (REPO / 'coreset_selection').is_dir():
    REPO = REPO.parent

csv_path = REPO / 'runs_out_quickstart' / 'nsga2-vae-popsoft-k30-rep0' / 'metrics.csv'
df = pd.read_csv(csv_path)
print(f'Rows: {len(df)}  Columns: {df.shape[1]}')

## 1. Group columns by metric family

There is no single formal schema — we partition by naming convention.

In [ ]:
families = {
    'identity':           ('coreset_name', 'k'),
    'geographic':         ('geo_',),
    'operator_fidelity':  ('nystrom_error', 'kpca_distortion'),
    'krr':                ('krr_',),
    'kpi_stability':      ('kpi_', 'state_kpi'),
    'multi_model':        ('knn_', 'rf_', 'lr_', 'gbt_'),
}

def classify(col):
    for fam, keys in families.items():
        if any(k in col for k in keys) or col in keys:
            return fam
    return 'other'

by_family = {}
for c in df.columns:
    by_family.setdefault(classify(c), []).append(c)

for fam, cols in by_family.items():
    print(f'{fam:<22} ({len(cols):>3} cols)')
    for c in cols[:6]:
        print(f'    {c}')
    if len(cols) > 6:
        print(f'    ... ({len(cols)-6} more)')

## 2. Meaning of the most important columns

| Column | Meaning | Lower or higher is better? |
|--------|---------|----------------------------|
| `nystrom_error` | `‖K_EE − K̂‖_F / ‖K_EE‖_F` where `K̂` is the Nyström approximation using the coreset as landmarks. | **lower** |
| `kpca_distortion` | Normed difference between top-r eigenvalues of `K_EE` and `K̂`. | **lower** |
| `krr_rmse_cov_area_4G` | Kernel ridge regression RMSE predicting 4G coverage from the coreset's Nyström features. | **lower** |
| `krr_rmse_cov_area_5G` | Same for 5G coverage. | **lower** |
| `geo_kl_pop` | KL divergence between target population-share and the coreset's realised population-share. | **lower** |
| `geo_maxdev_pop` | Max absolute state-share deviation under population-share weighting. | **lower** |
| `knn_rmse_*` / `rf_rmse_*` / `lr_*` / `gbt_*` | KNN/RF/LR/GBT performance on Nyström features for additional targets. | varies |

## 3. A small ranking exercise

Rank the Pareto-front members on operator fidelity and see which row wins.

In [ ]:
op_cols = [c for c in ('nystrom_error', 'kpca_distortion') if c in df.columns]
if op_cols:
    ranked = df[['coreset_name'] + op_cols].sort_values(op_cols[0]).head(5)
    print('Top 5 rows by nystrom_error:')
    print(ranked.to_string(index=False))

In [ ]:
# Per-metric minima — a practitioner might pick different coresets for different tasks.
numeric = df.select_dtypes(include='number')
lower_is_better = [c for c in numeric.columns if any(k in c for k in ('error', 'distortion', 'rmse', 'kl', 'maxdev'))]
print(f'For each of {len(lower_is_better)} "lower-is-better" metrics, the winning row:')
winners = {c: df.loc[df[c].idxmin(), 'coreset_name'] for c in lower_is_better[:10]}
for c, w in winners.items():
    print(f'  {c:<40} won by {w}')

Notice that *different coresets are the winners for different metrics* — this is the core insight that motivates the championship analysis: no single point on the Pareto front dominates every task.

## What's next

- [05_add_new_baseline.ipynb](./05_add_new_baseline.ipynb) — contribute a new baseline method.